In [18]:
import numpy as np
import pandas as pd

import joblib
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import PredefinedSplit, RandomizedSearchCV
from sklearn.impute import SimpleImputer
from sklearn.metrics import make_scorer, mean_absolute_error
from utils import *
from valid_models import valid_models

In [19]:
RANDOM_SEED = 1907
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

In [20]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

class FrequencyEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cols):
        self.cols = tuple(cols)

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        self.freq_values_ = {}
        for col in self.cols:
            freq_map = X[col].value_counts(normalize=True, dropna=False)
            mean_freq = float(freq_map.mean()) if len(freq_map) else 0.0
            self.freq_values_[col] = {"map": freq_map, "mean": mean_freq}
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        for col in self.cols:
            freq_map = self.freq_values_[col]["map"]
            mean_freq = self.freq_values_[col]["mean"]
            X[col] = X[col].map(freq_map).fillna(mean_freq).astype(float)
        return X


class InferBrand(BaseEstimator, TransformerMixin):
    def __init__(self, brand_col="Brand", model_col="model"):
        self.brand_col = brand_col
        self.model_col = model_col

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        self.model_to_brand_ = infer_brand_fit(X)  # your function
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        return infer_brand_apply(X, self.model_to_brand_)  # your function


class FillNaNs(BaseEstimator, TransformerMixin):
    def __init__(self, int_cols, float_cols):
        self.int_cols = tuple(int_cols)
        self.float_cols = tuple(float_cols)

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        _, self.fill_values_ = fill_nans(
            X, list(self.int_cols), list(self.float_cols), fill_values=None
        )
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        return fill_nans(
            X, list(self.int_cols), list(self.float_cols), fill_values=self.fill_values_
        )


class OutliersSkews(BaseEstimator, TransformerMixin):
    def __init__(self, num_cols, q_low=0.001, q_high=0.999, upper_only_after_log=False):
        self.num_cols = tuple(num_cols)
        self.q_low = q_low
        self.q_high = q_high
        self.upper_only_after_log = upper_only_after_log

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        _, self.info_ = outliers_skews_train(
            X,
            list(self.num_cols),
            q_low=self.q_low,
            q_high=self.q_high,
            upper_only_after_log=self.upper_only_after_log,
        )
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        return outliers_skews_test(X, self.info_)


In [21]:
class FrozenTransformer(BaseEstimator, TransformerMixin):
    """
    Wraps a PRE-FITTED transformer (e.g. a scaler) so that:
      - .fit() does NOTHING
      - .transform() delegates to the stored transformer

    ⚠️ WARNING:
    Using this inside cross-validation WILL cause data leakage
    unless the wrapped transformer was fitted ONLY on the
    corresponding training split.

    This is mainly useful for:
      - production inference
      - post-CV refits
      - non-CV workflows
    """
    def __init__(self, transformer):
        self.transformer = transformer

    def fit(self, X, y=None):
        # Intentionally do nothing
        return self

    def transform(self, X):
        return self.transformer.transform(X)


def _scaler_step(scaler, freeze=False, name="scaler"):
    """
    Decides how a scaler should be inserted into a Pipeline.

    Parameters
    ----------
    scaler : sklearn scaler or None
        The scaler TYPE you want to use (e.g. StandardScaler()).
    freeze : bool, default=False
        - False (recommended): clone(scaler) so sklearn fits it
          separately inside each CV fold (NO leakage).
        - True: wrap scaler in FrozenTransformer and reuse it as-is
          (⚠️ leakage risk in CV).
    name : str
        Step name used in the pipeline.

    Returns
    -------
    tuple or None
        A (name, transformer) pipeline step, or None if no scaler.
    """
    if scaler is None:
        return None

    if freeze:
        # Use the scaler exactly as provided (already fitted)
        return (name, FrozenTransformer(scaler))

    # Clone so each CV fold gets a FRESH, UNFITTED scaler
    return (name, clone(scaler))

In [22]:

def make_pipeline(
    *,
    int_cols,
    float_cols,
    cat_cols,
    encoding_type,     # "ohe" or "freq"
    estimator,
    scaler=None,       # pass StandardScaler(), RobustScaler(), or None
    brand_col="Brand",
    model_col="model",
    q_low=0.001,
    q_high=0.999,
    upper_only_after_log=False,
):
    num_cols = list(int_cols) + list(float_cols)

    # Numeric branch
    num_steps = [("imputer", SimpleImputer(strategy="mean"))]
    if scaler is not None:
        num_steps.append(("scaler", clone(scaler)))
    num_pipe = Pipeline(steps=num_steps)

    # Categorical branch
    if encoding_type == "ohe":
        cat_pipe = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ])
    elif encoding_type == "freq":
        cat_steps = [
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("freq", FrequencyEncoder(cat_cols)),
        ]
        # same scaler choice as numeric
        if scaler is not None:
            cat_steps.append(("scaler", clone(scaler)))
        cat_pipe = Pipeline(steps=cat_steps)
    else:
        raise ValueError("encoding_type must be 'ohe' or 'freq'")

    preprocess = ColumnTransformer(
        transformers=[
            ("num", num_pipe, num_cols),
            ("cat", cat_pipe, cat_cols),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )

    pipe = Pipeline(steps=[
        ("infer_brand", InferBrand(brand_col=brand_col, model_col=model_col)),
        ("fill_nans", FillNaNs(int_cols=int_cols, float_cols=float_cols)),
        ("outliers_skews", OutliersSkews(
            num_cols=num_cols,
            q_low=q_low,
            q_high=q_high,
            upper_only_after_log=upper_only_after_log,
        )),
        ("preprocess", preprocess),
        ("model", estimator),
    ])

    return pipe

In [23]:
def make_predefined_split(fold_id):
    fold_id = np.asarray(fold_id, dtype=int)
    return PredefinedSplit(test_fold=fold_id)

In [24]:
def make_pipeline(
    *,
    int_cols,
    float_cols,
    cat_cols,
    encoding_type,     # "ohe" or "freq"
    estimator,
    scaler=None,       # <-- you can pass StandardScaler(), RobustScaler(), or None
    brand_col="Brand",
    model_col="model",
    q_low=0.001,
    q_high=0.999,
    upper_only_after_log=False,
):
    num_cols = list(int_cols) + list(float_cols)

    # ------------------
    # Numeric branch
    # ------------------
    num_steps = [
        ("imputer", SimpleImputer(strategy="mean")),  # safety; FillNaNs already handles your logic
    ]
    if scaler is not None:
        num_steps.append(("scaler", clone(scaler)))   # clone => fresh scaler per CV fold

    num_pipe = Pipeline(steps=num_steps)

    # ------------------
    # Categorical branch
    # ------------------
    if encoding_type == "ohe":
        cat_pipe = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ])

    elif encoding_type == "freq":
        cat_steps = [
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("freq", FrequencyEncoder(cat_cols)),
        ]
        # ✅ same scaler choice as numeric branch
        if scaler is not None:
            cat_steps.append(("scaler", clone(scaler)))
        cat_pipe = Pipeline(steps=cat_steps)

    else:
        raise ValueError("encoding_type must be 'ohe' or 'freq'")

    preprocess = ColumnTransformer(
        transformers=[
            ("num", num_pipe, num_cols),
            ("cat", cat_pipe, cat_cols),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )

    # ✅ InferBrand BEFORE encoding categoricals
    pipe = Pipeline(steps=[
        ("infer_brand", InferBrand(brand_col=brand_col, model_col=model_col)),
        ("fill_nans", FillNaNs(int_cols=int_cols, float_cols=float_cols)),
        ("outliers_skews", OutliersSkews(
            num_cols=num_cols,
            q_low=q_low,
            q_high=q_high,
            upper_only_after_log=upper_only_after_log,
        )),
        ("preprocess", preprocess),
        ("model", estimator),
    ])

    return pipe

## Model #1 : Linear Regression (Selected Features, OHE Encoding)

Remove non important features according to our Feature Selection

In [25]:
df = pd.read_csv("train.csv")

num_cols = ['year', 'mileage', 'engineSize', 'power_efficiency']
cat_cols = ['model', 'transmission', "Brand"]
int_cols = ['year']
float_cols = ['mileage', 'engineSize', 'power_efficiency']

drop_cols = ['paintQuality%', 'previousOwners', 'fuelType', 'tax', 'mpg', 'mileage_per_year'] 

X = clean_df(df.copy(), valid_models, cat_cols)
X, y = separar_y(X)

X = X.drop(columns=drop_cols)
X

,Brand,model,year,transmission,mileage,engineSize,power_efficiency
carID,,,,,,,
69512,VW,GOLF,4,SEMI-AUTO,28421.0,2.0,0.175173
53000,Toyota,YARIS,1,MANUAL,4589.0,1.5,0.031315
6366,Audi,Q2,1,SEMI-AUTO,3624.0,1.5,0.036675
29021,Ford,FIESTA,2,MANUAL,9102.0,1.0,0.015221
10062,BMW,2SERIES,1,MANUAL,1000.0,1.5,0.035047
...,...,...,...,...,...,...,...
37194,Mercedes,CCLASS,5,MANUAL,14480.0,2.0,0.037523
6265,Audi,Q3,7,SEMI-AUTO,52134.0,2.0,0.041754
54886,Toyota,AYGO,3,AUTOMATIC,11304.0,1.0,0.014925


In [26]:
rng = np.random.default_rng(RANDOM_SEED)

#K-Fold Cross Validation
K = 5
fold_id = rng.integers(low=0, high=K, size=len(X), dtype=int)
ps = make_predefined_split(fold_id)


In [27]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

pipe = make_pipeline(
    int_cols=int_cols,
    float_cols=float_cols,
    cat_cols=cat_cols,
    encoding_type="ohe",
    scaler=StandardScaler(),            # used for numeric branch; OHE branch not scaled
    estimator=LinearRegression()
)

In [28]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer, mean_absolute_error
from sklearn.model_selection import cross_validate
from sklearn.metrics import make_scorer, mean_absolute_error

scoring = {
    "mae": make_scorer(mean_absolute_error, greater_is_better=False),
    "r2": "r2",
}

cv_results = cross_validate(
    estimator=pipe,
    X=X,
    y=y,
    cv=ps,          # ✅ your PredefinedSplit
    scoring=scoring,
    return_train_score=False,
)

mae_scores = -cv_results["test_mae"]   # negate to get actual MAE
r2_scores  = cv_results["test_r2"]

print("MAE per fold:", mae_scores)
print("Mean MAE:", mae_scores.mean())

print("R² per fold:", r2_scores)
print("Mean R²:", r2_scores.mean())

pipe.fit(X, y)

MAE per fold: [2492.60073114 2514.70019057 2548.18926312 2537.67266848 2495.88769719]
Mean MAE: 2517.810110101439
R² per fold: [0.83733109 0.82841174 0.81344154 0.83183234 0.83687611]
Mean R²: 0.8295785617426196


,steps,"[('infer_brand', ...), ('fill_nans', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,brand_col,'Brand'
,model_col,'model'
,int_cols,"('year',)"
,float_cols,"('mileage', ...)"
,num_cols,"('year', ...)"
,q_low,0.001
,q_high,0.999


In [29]:
test_df = pd.read_csv("test.csv")
X_test = clean_df(test_df.copy(), valid_models, cat_cols)
X_test = X_test.drop(columns=drop_cols)
XcarID=X_test.copy()

bundle = {
    "pipeline": pipe,
    "cat_cols": cat_cols,
    "int_cols": int_cols,
    "float_cols": float_cols,
    "encoding_type": "ohe",
}
joblib.dump(bundle, "linear_bundle.joblib")

y_pred = bundle["pipeline"].predict(X_test)

y_pred = pipe.predict(X_test)


In [30]:
submission = pd.DataFrame({
    "carID": XcarID.index,   # or test_df["carID"] if that exists
    "price": y_pred
})

display(submission.head())

# --- Save to CSV (with header, no index) ---
submission.to_csv("kaggle/model1.csv", index=False)

,carID,price
0,89856,12862.627236
1,106581,21793.884861
2,80886,12011.373405
3,100174,17696.485127
4,81376,23145.112635


## Model #2 : Random Forests (Selected Features, OHE Encoding)

Our feature Selection was a complete blunder when it comes to trees models, probably because the random forests in the feature selection part were trained fast and with leakage, we simply picked the best features manually. This had to be done unfortunately.

In [31]:
num_cols = ['year', 'mileage', 'tax', 'mpg', 'engineSize', 'mileage_per_year', 'power_efficiency', 'previousOwners', "mpg", "tax"]
cat_cols = ['Brand', 'model', 'transmission']
int_cols = ['year']

drop_cols = ['paintQuality%', 'previousOwners', 'fuelType']

df=pd.read_csv("train.csv")
dfcopy = df.copy()
X=clean_df(df, valid_models, cat_cols)
X, y = separar_y(X)
X = X.drop(columns=drop_cols)

In [32]:
rng = np.random.default_rng(RANDOM_SEED)

#K-Fold Cross Validation
K = 5
fold_id = rng.integers(low=0, high=K, size=len(X), dtype=int)
ps = make_predefined_split(fold_id)


In [ ]:
from sklearn.ensemble import RandomForestRegressor
scoring = {
    "mae": make_scorer(mean_absolute_error, greater_is_better=False),  # negative MAE
    "r2": "r2",
}

In [ ]:
"""
pipe = make_pipeline(
    int_cols=int_cols,
    float_cols=float_cols,
    cat_cols=cat_cols,
    encoding_type="ohe",
    scaler=None,            # used for numeric branch; OHE branch not scaled
    estimator=RandomForestRegressor(random_state=RANDOM_SEED, n_jobs=-1)
)

param_dist = {
    "model__n_estimators": [50, 75, 100, 150, 200, 300, 400, 500, 600],
    "model__max_depth": [5, 7, 10, 15, 20, 25, 30],
    "model__min_samples_split": [2, 4, 5, 10, 12, 15],
    "model__min_samples_leaf": [1, 2, 5, 7, 10],
    "model__max_features": ["sqrt", "log2", 0.8, 0.5, 0.7],
    "model__max_samples": [0.7, 0.8, 0.9],
    "model__bootstrap": [True],
}

search = RandomizedSearchCV(
    estimator=pipe,
    param_distributions=param_dist,
    n_iter=1,
    scoring=scoring,     # compute both
    refit="mae",         # ✅ choose best params by MAE ONLY
    cv=ps,               # ✅ PredefinedSplit
    random_state=RANDOM_SEED,
    n_jobs=1,
    verbose=3,
    return_train_score=True,
)

search.fit(X, y)
"""

In [48]:
best_params = {"random_state": RANDOM_SEED,
  "n_jobs": -1,
  "n_estimators": 75,
  "max_depth": 30,
  "min_samples_split": 4,
  "min_samples_leaf": 1,
  "max_features": 0.7,
  "max_samples": 0.8,
  "bootstrap": True}

In [50]:
pipe = make_pipeline(
    int_cols=int_cols,
    float_cols=float_cols,
    cat_cols=cat_cols,
    encoding_type="ohe",
    scaler=None,            # used for numeric branch; OHE branch not scaled
    estimator=RandomForestRegressor(**best_params)
)
cv_results = cross_validate(
    estimator=pipe,
    X=X,
    y=y,
    cv=ps,          # ✅ your PredefinedSplit
    scoring=scoring,
    return_train_score=False,
)

mae_scores = -cv_results["test_mae"]   # negate to get actual MAE
r2_scores  = cv_results["test_r2"]

print("MAE per fold:", mae_scores)
print("Mean MAE:", mae_scores.mean())

print("R² per fold:", r2_scores)
print("Mean R²:", r2_scores.mean())

pipe.fit(X, y)

MAE per fold: [1416.42433547 1399.21442088 1416.25402618 1408.80156879 1368.26349811]
Mean MAE: 1401.791569886915
R² per fold: [0.9359537  0.93771007 0.92442404 0.93802347 0.93945259]
Mean R²: 0.9351127750826642


,steps,"[('infer_brand', ...), ('fill_nans', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,brand_col,'Brand'
,model_col,'model'
,int_cols,"('year',)"
,float_cols,"('mileage', ...)"
,num_cols,"('year', ...)"
,q_low,0.001
,q_high,0.999


In [51]:
test_df = pd.read_csv("test.csv")
X_test = clean_df(test_df.copy(), valid_models, cat_cols)
X_test = X_test.drop(columns=drop_cols)
XcarID=X_test.copy()

bundle = {
    "pipeline": pipe,
    "cat_cols": cat_cols,
    "int_cols": int_cols,
    "float_cols": float_cols,
    "encoding_type": "ohe",
}
joblib.dump(bundle, "linear_bundle.joblib")

y_pred = bundle["pipeline"].predict(X_test)

y_pred = pipe.predict(X_test)


In [53]:
submission = pd.DataFrame({
    "carID": XcarID.index,   # or test_df["carID"] if that exists
    "price": y_pred
})

display(submission.head())

# --- Save to CSV (with header, no index) ---
submission.to_csv("kaggle/model2.csv", index=False)

,carID,price
0,89856,12261.171763
1,106581,24388.479664
2,80886,13640.002000
3,100174,16615.312190
4,81376,25094.977837
